# Exercise 09 — Retrieval-Augmented Generation

In notebook 1, the generative model answered much better when it was given the right passage (*open-book*) than from memory alone (*closed-book*), where it confidently made things up. But in practice nobody hands the model the right passage. **Retrieval-augmented generation** (RAG; [Lewis et al., 2020](https://arxiv.org/abs/2005.11401)) automates the open-book setting in two steps:

1. **Retrieve**: given the question, find the most relevant passages in a large collection of documents.
2. **Generate**: give the question *and* the retrieved passages to a language model, and let it answer from them.

RAG lets a model use knowledge it never saw during training (recent events, a company's internal documents, your course notes), makes its answers checkable against their sources, and reduces hallucinations, without retraining the model. Most chat assistants with "search" work this way.

## What you will do
1. **Why retrieval?**: an LLM that does not know the answer, and one that is given the documents.
2. **A retrieval benchmark**: the SQuAD paragraphs as a document collection, and Recall@k.
3. **BM25**: the classic keyword-based ranking function, implemented by hand.
4. **Dense retrieval**: sentence embeddings with mean pooling, and combining both retrievers.
5. **RAG**: prompting an LLM with retrieved passages, compared with closed-book and with the gold passage.
6. **LLM-as-a-judge**: letting an LLM grade the answers, and how far to trust it.
7. **Multiple-choice questions**: to check your understanding.

## How to work through it
- Run the cells **in order** and fill in **every `# TODO`**.
- Tasks are numbered (**1.1**, **1.2**, …). Tasks that say *Your answer here* want a short written answer, not code.
- Most implementation tasks are followed by a **✅ Check** cell that verifies your work automatically. Run it and make sure it passes before you move on.

> **Heads up: this notebook calls an online API**, the same Groq API as in week 7, so you need your free Groq API key. It also downloads the sentence-embedding model `sentence-transformers/all-MiniLM-L6-v2` (~90 MB) and the SQuAD validation set (~10 MB). No GPU is needed: embedding the documents in Part 4 takes about a minute on a CPU. The LLM experiments use roughly 50,000 tokens of the free daily budget (as of October 2026: 200,000 tokens per day for the model we use) and take a few minutes because of the per-minute limit. Every reply is cached in `groq_cache.json` next to the notebook, so running a cell a second time costs nothing.

In [76]:
import hashlib
import json
import math
import os
import re
import string
from collections import Counter, defaultdict
from getpass import getpass

import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from groq import Groq
from transformers import AutoModel, AutoTokenizer

# Use the GPU if there is one; a CPU is enough for this notebook.
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using mps device


## 1. Why retrieval?

We use the same setup as in week 7: a 27 B parameter model on Groq, and a cached `chat` function. Run the cell and paste your API key when you are asked for it.

> **Keep your key secret.** Never write it into a code cell. If you set the environment variable `GROQ_API_KEY` before starting Jupyter, the cell does not ask at all.

In [77]:
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key: ")

client = Groq(max_retries=8)  # reads GROQ_API_KEY; on a rate-limit error it waits and tries again (up to 8 times)

# A 27 B parameter open-weights model. If it is no longer available, pick another one from
# https://console.groq.com/docs/models and check its rate limits.
MODEL = "qwen/qwen3.8-27b"

CACHE_PATH = "groq_cache.json"
try:
    with open(CACHE_PATH, encoding="utf-8") as f:
        cache = json.load(f)
except FileNotFoundError:
    cache = {}


def chat(messages, temperature=0.0, max_tokens=256):
    # Send a conversation to the model; returns the reply (a string) and the number of tokens used.
    key = hashlib.sha256(json.dumps([MODEL, messages, temperature, max_tokens]).encode()).hexdigest()
    if key not in cache:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=temperature,
            max_completion_tokens=max_tokens,
            reasoning_effort="none",
        )
        cache[key] = {"reply": response.choices[0].message.content, "tokens": response.usage.total_tokens}
        with open(CACHE_PATH, "w", encoding="utf-8") as f:
            json.dump(cache, f)
    return cache[key]["reply"], cache[key]["tokens"]


def ask(question):
    return chat([{"role": "user", "content": question}])[0]

Here is a small collection of internal documents of an (invented) bike shop in Aarhus. The LLM cannot possibly have seen them during training.

In [78]:
handbook = [
    "Fjordbyen Cykler was founded in 2011 by Mette Holm and has two shops in Aarhus, one in Frederiksbjerg and one in Trøjborg.",
    "New employees at Fjordbyen Cykler get 27 days of paid holiday per year, two more than the Danish minimum.",
    "Customers can return a bike within 30 days if it has not been ridden more than 50 km. Repairs come with a 6-month guarantee.",
    "The Trøjborg shop is closed on Mondays. Both shops are open until 20:00 on Thursdays.",
    "Fjordbyen Cykler offers students a 15 % discount on repairs, but not on new bikes.",
    "The staff Christmas party is always held on the first Friday of December at the Frederiksbjerg shop.",
]
question = "How many days of paid holiday do new employees at Fjordbyen Cykler get?"

print("CLOSED-BOOK:", ask(question))

CLOSED-BOOK: Based on standard Norwegian labor laws and typical employment practices for companies like **Fjordbyen Cykler** (a bicycle shop/retail business in Norway), here is the breakdown:

### 1. **Standard Norwegian Paid Holiday Entitlement**
In Norway, the **Working Environment Act (Arbeidsmiljøloven)** mandates that all employees are entitled to **25 days of paid holiday** per year. This is the legal minimum and applies to new employees from their first day of employment.

### 2. **Accrual for New Employees**
- New employees **do not** receive 25 days of holiday immediately in their first year.
- Holiday is **accrued** over time. Typically, employees earn **2 days of holiday for every month of work**.
- Therefore, a new employee will gradually build up their holiday entitlement during their first year.
- By the end of their first full year of employment, they will have accrued the full **25 days**.

### 3. **Company-Specific Policies**
While 25 days is the legal standard, some c

**1.1 Give the model the documents.** Write a prompt that contains all the documents of the handbook and the question, and ask the model to answer from the documents.

In [79]:
prompt = f"""
Answer the question using the following documents:
{"\n".join(f"- {d}" for d in handbook)}

Question: {question}
"""  # TODO: the documents and the question in one prompt

print("WITH THE DOCUMENTS:", ask(prompt))

WITH THE DOCUMENTS: New employees at Fjordbyen Cykler get **27 days** of paid holiday per year.


In [80]:
# ✅ Check your prompt
assert isinstance(prompt, str) and all(doc in prompt for doc in handbook) and question in prompt, "the prompt should contain every document and the question"
assert "27" in ask(prompt), "with the documents, the model should answer 27 days"
print("Looks good ✅")

Looks good ✅


**1.2 Closed-book vs. with the documents.** What did the model answer closed-book: did it admit that it does not know, or did it guess? Putting *all* documents into the prompt worked here. Why does that not work for the documents of a real company, or for all of Wikipedia? Name at least two reasons.

---

*Your answer here:*

---

## 2. A retrieval benchmark

To compare retrievers, we need questions for which we know the relevant document. SQuAD (notebook 1) gives us exactly that: we treat its **2,067 validation paragraphs** as the document collection, and every question's own paragraph as the one relevant document. The retriever has to find it among all others, which come from 48 Wikipedia articles, so many paragraphs are about the same topic.

The standard measure is **Recall@k**: the fraction of questions for which the relevant document is among the top *k* retrieved documents. For RAG it is the number that matters most: if the right passage is not among the *k* passages in the prompt, the generator cannot read the answer from it.

In [81]:
squad = load_dataset("rajpurkar/squad", split="validation")

documents = list(dict.fromkeys(list(squad["context"])))  # the unique paragraphs, in order
doc_index = {doc: i for i, doc in enumerate(documents)}
print(f"{len(squad)} questions, {len(documents)} documents")

N_QUERIES = 500
queries = squad.shuffle(seed=0).select(range(N_QUERIES))
gold = np.array([doc_index[ex["context"]] for ex in queries])  # the index of the relevant document of every query
print(queries[0]["question"], "->", gold[0])

10570 questions, 2067 documents
How many US states do no practice corporal punishment? -> 305


**2.1 Complete `recall_at_k`.** It gets a matrix of retrieval scores, one row per query and one column per document, and returns the fraction of queries whose gold document is among the `k` highest-scoring documents.

Hint: `np.argsort(-scores, axis=1)[:, :k]` gives the indices of the `k` highest scores of every row.

In [82]:
def recall_at_k(scores, gold, k):
    # The fraction of queries (rows of scores) whose gold document is among the top-k documents
    top_k = np.argsort(-scores, axis=1)[:, :k]  # TODO: (queries, k) indices of the highest-scoring documents
    #print(f"{top_k = }\n{gold = }\n{(top_k == gold[:, None]).any(axis=1).mean()}")
    return (top_k == gold[:, None]).any(axis=1).mean()   # TODO

In [83]:
# ✅ Check recall_at_k
toy_scores = np.array([[0.1, 0.9, 0.3], [0.8, 0.1, 0.5], [0.2, 0.3, 0.1]])
toy_gold = np.array([1, 2, 2])
assert recall_at_k(toy_scores, toy_gold, 1) == 1 / 3, "only the first query has its gold document at rank 1"
assert recall_at_k(toy_scores, toy_gold, 2) == 2 / 3
assert recall_at_k(toy_scores, toy_gold, 3) == 1.0
print("Looks good ✅")

Looks good ✅


## 3. BM25

**BM25** ([Robertson & Zaragoza, 2009](https://www.staff.city.ac.uk/~sbrp622/papers/foundations_bm25_review.pdf)) has been the standard ranking function of search engines for decades, and it is still a strong baseline. Like TF-IDF, it treats documents and queries as **bags of words** and scores a document higher if it contains the query's words, especially rare ones. For a query $q$ and a document $d$:

$$\text{BM25}(q, d) = \sum_{t \in q} \text{idf}(t) \cdot \frac{f_{t,d}\,(k_1 + 1)}{f_{t,d} + k_1 \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)} \qquad \text{idf}(t) = \ln\left(1 + \frac{N - n_t + 0.5}{n_t + 0.5}\right)$$

- $f_{t,d}$ is how often term $t$ occurs in $d$, $|d|$ the length of $d$ in tokens, and avgdl the average length of all documents.
- $N$ is the number of documents and $n_t$ the number of documents that contain $t$. Rare terms get a high idf; a term in almost every document gets an idf close to 0.
- $k_1$ (typically 1.2–2) controls **term-frequency saturation**: the second occurrence of a word adds less than the first, and the score for a term can never exceed $\text{idf}(t) \cdot (k_1 + 1)$, however often it occurs.
- $b$ (typically 0.75) controls **length normalisation**: a word occurrence counts for less in a long document than in a short one.

A search engine does not compute the score for every document from scratch. It builds an **inverted index**: for every term, the list of documents that contain it, with the term frequency. To score a query, it only visits the lists of the query's terms. The class below builds the index.

**3.1 Complete `idf` and `scores`.**

In [84]:
STOPWORDS = set(
    "a an and are as at be by for from has have he her his how in is it its of on or she that the their they this "
    "to was were what when where which who whom whose why will with did does do".split()
)


def bm25_tokenize(text):
    # Lower-case words and numbers, without stopwords
    return [token for token in re.findall(r"[a-z0-9]+", text.lower()) if token not in STOPWORDS]


class BM25:
    def __init__(self, documents, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        doc_tokens = [bm25_tokenize(doc) for doc in documents]
        self.N = len(documents)
        self.doc_len = np.array([len(tokens) for tokens in doc_tokens])
        self.avgdl = self.doc_len.mean()
        # The inverted index: term -> list of (document index, term frequency)
        self.postings = defaultdict(list)
        for i, tokens in enumerate(doc_tokens):
            for term, tf in Counter(tokens).items():
                self.postings[term].append((i, tf))

    def idf(self, term):
        n_t = len(self.postings[term]) if term in self.postings else 0
        nom = self.N - n_t + 0.5
        denom = n_t + 0.5
        return np.log(1 + nom / denom)  # TODO

    def scores(self, query):
        # The BM25 score of every document for the query: a numpy array of shape (N,)
        scores = np.zeros(self.N)
        for term in bm25_tokenize(query):
            for i, tf in self.postings.get(term, []):
                nom = tf * (self.k1 + 1)
                denom = tf + self.k1 * (1 - self.b + self.b * self.doc_len[i] / self.avgdl)
                scores[i] += self.idf(term) * nom / denom  # TODO
        return scores


bm25 = BM25(documents)
print(f"{len(bm25.postings):,} terms in the index, average document length {bm25.avgdl:.1f} tokens")

22,872 terms in the index, average document length 83.3 tokens


In [85]:
# ✅ Check BM25 on a tiny collection
toy = BM25(["cats purr", "dogs bark loudly at cats", "dogs dogs dogs everywhere", "birds sing"])
assert abs(toy.idf("cats") - np.log(1 + 2.5 / 2.5)) < 1e-9, "'cats' occurs in 2 of 4 documents"
assert abs(toy.idf("zebra") - np.log(1 + 4.5 / 0.5)) < 1e-9, "a term that occurs nowhere has n_t = 0"
expected = np.array([0.0, 0.60274, 1.06638, 0.0])
assert np.allclose(toy.scores("dogs"), expected, atol=1e-4), f"scores for 'dogs': yours {toy.scores('dogs')}, expected {expected}"
assert np.allclose(toy.scores("the dogs"), expected, atol=1e-4), "stopwords should not change the score"
assert np.allclose(toy.scores("dogs cats"), toy.scores("dogs") + toy.scores("cats")), "the score is a sum over the query terms"
print("Looks good ✅")

Looks good ✅


**3.2 Evaluate BM25.** Compute the scores of all queries (a matrix of shape `(N_QUERIES, documents)`) and the Recall@k for k = 1, 3, 5 and 20.

In [86]:
bm25_scores = np.stack([bm25.scores(q["question"]) for q in queries])  # TODO: (N_QUERIES, len(documents))

for k in [1, 3, 5, 20]:
    print(f"BM25 Recall@{k:<2} {recall_at_k(bm25_scores, gold, k):.3f}")

BM25 Recall@1  0.770
BM25 Recall@3  0.894
BM25 Recall@5  0.912
BM25 Recall@20 0.950


In [87]:
# ✅ Check bm25_scores
assert bm25_scores.shape == (N_QUERIES, len(documents)), "one row per query, one column per document"
assert recall_at_k(bm25_scores, gold, 5) > 0.6, "BM25 should find the gold paragraph in the top 5 for most SQuAD questions"
print("Looks good ✅")

Looks good ✅


The cell below shows some questions for which the gold paragraph was **not** in the top 5.

In [88]:
misses = [i for i in range(N_QUERIES) if gold[i] not in np.argsort(-bm25_scores[i])[:5]]
for i in misses[:5]:
    top = int(np.argmax(bm25_scores[i]))
    print(f"Q: {queries[i]['question']}   (answer: {queries[i]['answers']['text'][0]!r})")
    print(f"   retrieved: {documents[top][:150]}...")
    print(f"   gold:      {documents[gold[i]][:150]}...\n")

Q: How did Luther's writings sound as he became less healthy?   (answer: 'harsher')
   retrieved: Early in 1537, Johannes Agricola (1494–1566) – serving at the time as pastor in Luther's birthplace, Eisleben – preached a sermon in which he claimed ...
   gold:      His poor physical health made him short-tempered and even harsher in his writings and comments. His wife Katharina was overheard saying, "Dear husband...

Q: What is the name for O3 most often used?   (answer: 'ozone')
   retrieved: The Rhine-Meuse Delta, the most important natural region of the Netherlands begins near Millingen aan de Rijn, close to the Dutch-German border with t...
   gold:      Trioxygen (O
3) is usually known as ozone and is a very reactive allotrope of oxygen that is damaging to lung tissue. Ozone is produced in the upper a...

Q: Who were the defending Super Bowl champions?   (answer: 'New England Patriots')
   retrieved: In addition to the Vince Lombardi Trophy that all Super Bowl champions receive, t

**3.3 BM25's strengths and weaknesses.** SQuAD's questions were written by crowdworkers *while looking at the paragraph*. How does that help BM25? Look at the misses: why did BM25 fail on them? Give an example of a question that BM25 would find hard even though the answer is in a paragraph.

---

*Your answer here:*

---

## 4. Dense retrieval

**Dense retrieval** maps queries and documents to **vectors** with a neural encoder, so that a question and a passage that answers it are close in the vector space, even if they share no words, and ranks the documents by their cosine similarity with the query ([Karpukhin et al., 2020](https://arxiv.org/abs/2004.04906)). The encoder is trained on (question, relevant passage) pairs with a contrastive loss: matching pairs are pulled together, non-matching ones pushed apart.

We use [`sentence-transformers/all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2), a small (22 M parameters, 6 layers) encoder that was trained in this way on over a billion sentence pairs. It produces one vector per *token*; a single vector for the whole text is obtained by **mean pooling**: the average of the token vectors, ignoring padding. Then the vector is normalised to length 1, so that a dot product is a cosine similarity.

**4.1 Complete the mean pooling in `embed`.**

Hint: multiply the token vectors by the attention mask (with an extra dimension, `mask[:, :, None]`), sum over the tokens, and divide by the number of real tokens. With padding, a plain `.mean(dim=1)` would include the padding tokens.

In [89]:
EMB_NAME = "sentence-transformers/all-MiniLM-L6-v2"

emb_tokenizer = AutoTokenizer.from_pretrained(EMB_NAME)
emb_model = AutoModel.from_pretrained(EMB_NAME).to(device)
emb_model.eval()


def embed(texts, batch_size=64):
    # One unit-length vector per text: a tensor of shape (len(texts), 384)
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = emb_tokenizer(texts[i : i + batch_size], padding=True, truncation=True, max_length=256, return_tensors="pt").to(device)
        with torch.no_grad():
            hidden = emb_model(**batch).last_hidden_state  # (batch, tokens, 384)
        mask = batch["attention_mask"].float()               # (batch, tokens)
        masked_hidden = (hidden * mask[:, :, None]).sum(dim=1)
        mask_sum = mask.sum(dim=1, keepdim=True)
        #print(f"{masked_hidden = }\n{mask_sum = }")
        pooled = masked_hidden / mask_sum  # TODO: the mean of the token vectors of every text, without the padding
        vectors.append(F.normalize(pooled, dim=-1).cpu())
    return torch.cat(vectors)


v = embed(["Which illness killed the composer?", "Chopin succumbed to tuberculosis.", "The Broncos won the Super Bowl."])
print(v.shape)
print((v[0] @ v.T).tolist())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

torch.Size([3, 384])
[1.0, 0.5415850877761841, -0.0010841134935617447]


In [90]:
# ✅ Check embed
short, long = "Chopin died young.", "Frédéric Chopin was a Polish composer and virtuoso pianist of the Romantic period who wrote mainly for the piano."
alone = embed([short])
padded = embed([short, long])[0:1]
assert alone.shape == (1, 384) and torch.allclose(alone.norm(dim=-1), torch.ones(1), atol=1e-5), "the vectors should have length 1"
assert torch.allclose(alone, padded, atol=1e-4), "the embedding of a text should not depend on the padding in its batch"
print("Looks good ✅")

Looks good ✅


**4.2 Evaluate dense retrieval.** Embed all documents (this takes about a minute on a CPU) and the queries, compute the score matrix of cosine similarities, and compare its Recall@k with BM25.

In [91]:
doc_vectors = embed(documents)
query_vectors = embed(list(queries["question"]))  # TODO
dense_scores = (query_vectors @ doc_vectors.T).numpy()   # TODO: (N_QUERIES, len(documents)) cosine similarities, as a numpy array

print(f"{'':6} {'BM25':>6} {'dense':>6}")
for k in [1, 3, 5, 20]:
    print(f"R@{k:<4} {recall_at_k(bm25_scores, gold, k):6.3f} {recall_at_k(dense_scores, gold, k):6.3f}")

         BM25  dense
R@1     0.770  0.628
R@3     0.894  0.814
R@5     0.912  0.866
R@20    0.950  0.958


In [92]:
# ✅ Check dense_scores
assert isinstance(dense_scores, np.ndarray) and dense_scores.shape == (N_QUERIES, len(documents)), "a numpy array with one row per query"
assert dense_scores.max() <= 1 + 1e-5 and dense_scores.min() >= -1 - 1e-5, "cosine similarities lie between -1 and 1"
assert recall_at_k(dense_scores, gold, 20) > 0.6
print("Looks good ✅")

Looks good ✅


### Combining the two

BM25 and dense retrieval make different mistakes, so combining them often beats both. Their scores live on different scales (BM25 scores are unbounded, cosine similarities lie between -1 and 1), so instead of adding scores we combine **ranks**. **Reciprocal rank fusion** ([Cormack et al., 2009](https://dl.acm.org/doi/10.1145/1571941.1572114)) gives every document the score

$$\text{RRF}(d) = \sum_{\text{retrievers } r} \frac{1}{c + \text{rank}_r(d)}$$

where $\text{rank}_r(d)$ is the position of $d$ in the ranking of retriever $r$ (1 for the best document), and $c = 60$ dampens the influence of the very top ranks.

**4.3 Complete `ranks` and `reciprocal_rank_fusion`.**

Hint: `np.argsort(np.argsort(-scores, axis=1), axis=1)` gives, for every entry, its position in the descending order of its row, starting at 0.

In [93]:
def ranks(scores):
    # The rank (1 = best) of every document for every query, same shape as scores
    return np.argsort(np.argsort(-scores, axis=1), axis=1) + 1  # TODO


def reciprocal_rank_fusion(*score_matrices, c=60):
    return sum(1 / (c + ranks(s)) for s in score_matrices)  # TODO: the sum over the score matrices of 1 / (c + rank)


hybrid_scores = reciprocal_rank_fusion(bm25_scores, dense_scores)
print(f"{'':6} {'BM25':>6} {'dense':>6} {'hybrid':>6}")
for k in [1, 3, 5, 20]:
    print(f"R@{k:<4} {recall_at_k(bm25_scores, gold, k):6.3f} {recall_at_k(dense_scores, gold, k):6.3f} {recall_at_k(hybrid_scores, gold, k):6.3f}")

         BM25  dense hybrid
R@1     0.770  0.628  0.740
R@3     0.894  0.814  0.882
R@5     0.912  0.866  0.936
R@20    0.950  0.958  0.982


In [94]:
# ✅ Check ranks and reciprocal_rank_fusion
assert (ranks(np.array([[0.2, 0.9, 0.5]])) == np.array([[3, 1, 2]])).all(), "the highest score has rank 1"
a, b = np.array([[3.0, 2.0, 1.0]]), np.array([[0.1, 0.3, 0.2]])
assert np.allclose(reciprocal_rank_fusion(a, b), [[1 / 61 + 1 / 63, 1 / 62 + 1 / 61, 1 / 63 + 1 / 62]])
assert hybrid_scores.shape == bm25_scores.shape
print("Looks good ✅")

Looks good ✅


**4.4 Which retriever would you use?** Compare the three columns. Which retriever is better at k = 1, and which at k = 20? Why might dense retrieval do comparatively *better* on questions written by real users than on SQuAD? And what kind of query would a dense retriever get wrong that BM25 gets right? (Think of product codes, rare names, or numbers.)

---

*Your answer here:*

---

## 5. Retrieval-augmented generation

Now we put the pieces together. For a question, we retrieve the top-`k` paragraphs and give them to the LLM. We compare four settings on `N_RAG = 20` questions:

- **closed-book**: the question only,
- **BM25** and **hybrid**: the top-3 paragraphs of the retriever,
- **oracle**: the gold paragraph. This is the upper bound for what a perfect retriever would give.

The metrics are the ones from notebook 1:

In [95]:
def normalize_answer(text):
    text = "".join(ch for ch in text.lower() if ch not in set(string.punctuation))
    return " ".join(re.sub(r"\b(a|an|the)\b", " ", text).split())


def exact_match(prediction, references):
    return float(any(normalize_answer(prediction) == normalize_answer(r) for r in references))


def f1_score(prediction, references):
    def f1(pred, ref):
        pred, ref = normalize_answer(pred).split(), normalize_answer(ref).split()
        if not pred or not ref:
            return float(pred == ref)
        common = sum((Counter(pred) & Counter(ref)).values())
        return 0.0 if common == 0 else 2 * common / (len(pred) + len(ref))

    return max(f1(prediction, r) for r in references)


def contains_answer(prediction, references):
    return float(any(f" {normalize_answer(r)} " in f" {normalize_answer(prediction)} " for r in references))

**5.1 Complete `rag_messages`.** The prompt should:
- contain the passages, numbered (`[1] ...`, `[2] ...`), and the question,
- tell the model to answer **only from the passages**, with a **short answer** of a few words,
- tell it to reply `unanswerable` if the passages do not contain the answer.

With no passages, ask the question closed-book (with the same instruction to answer briefly).

In [96]:
def rag_messages(question, passages):
    # The conversation for a question and a list of passages (closed-book if the list is empty)
    if not passages:
        return [
            {"role": "system", "content": "Answer the question with a short answer of a few words, not a full sentence."},
            {"role": "user", "content": question},
        ]
    system = """
    Answer the question using only the information in the passages.
    Reply with a short answer of a few words, not a full sentence.
    If the passages do not contain the answer, reply 'unanswerable'.
    """    # TODO: answer only from the passages, briefly, or reply "unanswerable"
    context = "\n\n".join(f"[{i}] {passage}" for i, passage in enumerate(passages, start=1))   # TODO: the numbered passages
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": f"{context}\n\nQuestion: {question}"},
    ]


def rag_answer(question, passages):
    return chat(rag_messages(question, passages), max_tokens=64)[0].strip()


print(rag_answer("How many days of paid holiday do new employees at Fjordbyen Cykler get?", handbook[:3]))
print(rag_answer("When was Aarhus University founded?", handbook[:3]))

27 days
unanswerable


In [97]:
# ✅ Check rag_messages
messages = rag_messages("What is the question?", ["first passage", "second passage"])
text = " ".join(m["content"] for m in messages)
assert "[1] first passage" in text and "[2] second passage" in text, "number the passages as [1], [2], ..."
assert "What is the question?" in text and "unanswerable" in text, "include the question and the instruction to reply 'unanswerable'"
assert "unanswerable" in rag_answer("When was Aarhus University founded?", handbook[:3]).lower(), "the handbook does not say when Aarhus University was founded"
print("Looks good ✅")

Looks good ✅


**5.2 Run the comparison.** For each of the first `N_RAG` queries, get an answer in each of the four settings, and print EM, F1 and containment per setting.

Hint: `np.argsort(-scores[i])[:K]` gives the indices of the top-`K` documents of query `i`.

In [98]:
N_RAG, K = 20, 3

settings = {"closed-book": [], "BM25": [], "hybrid": [], "oracle": []}
for i in range(N_RAG):
    question = queries[i]["question"]
    settings["closed-book"].append(rag_answer(question, []))
    settings["BM25"].append(rag_answer(question, [documents[j] for j in np.argsort(bm25_scores[i], descending=True)[:k]]))    # TODO: the top-K documents of BM25
    settings["hybrid"].append(rag_answer(question, [documents[j] for j in np.argsort(hybrid_scores[i], descending=True)[:k]]))  # TODO: the top-K documents of the hybrid retriever
    settings["oracle"].append(rag_answer(question, [documents[gold[i]]]))  # TODO: the gold document only

references = [queries[i]["answers"]["text"] for i in range(N_RAG)]
print(f"{'':12} {'EM':>5} {'F1':>5} {'contains':>9}")
for name, answers in settings.items():
    em = 100 * np.mean([exact_match(a, r) for a, r in zip(answers, references)])
    f1 = 100 * np.mean([f1_score(a, r) for a, r in zip(answers, references)])
    contains = 100 * np.mean([contains_answer(a, r) for a, r in zip(answers, references)])
    print(f"{name:12} {em:5.1f} {f1:5.1f} {contains:9.1f}")

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3.8-27b` in organization `org_01m4ttsv56ebbr3krz5dqgp7my` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199802, Requested 3044. Please try again in 20m29.472s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
# ✅ Check the comparison
assert all(len(answers) == N_RAG and all(isinstance(a, str) for a in answers) for answers in settings.values()), "one answer per question and setting"
assert np.mean([f1_score(a, r) for a, r in zip(settings["oracle"], references)]) > 0.5, "with the gold passage, the model should get most answers right"
print("Looks good ✅")

In [ ]:
for i in range(N_RAG):
    answers = {name: settings[name][i] for name in settings}
    if len({normalize_answer(a) for a in answers.values()}) > 1:
        print(f"Q: {queries[i]['question']}   reference: {references[i][0]!r}")
        for name, answer in answers.items():
            print(f"   {name:12} {answer!r}")
        print()

**5.3 What does retrieval add?**
1. How does closed-book compare with the RAG settings here? SQuAD is built from Wikipedia, which the LLM has read during pretraining; why can it still fail closed-book? Would you expect a bigger or smaller gap for the handbook from Part 1?
2. How far is the RAG system from the oracle? When a RAG answer is wrong, how can you tell whether the **retriever** or the **generator** is to blame? (Look at the Recall@3 of the retriever.)
3. With more passages (`K = 10`), the chance that the right one is among them goes up. What could go *down*?

---

*Your answer here:*

---

## 6. LLM-as-a-judge

In notebooks 0 and 1 we saw that string-matching metrics are unfair to generated answers, and that people are the gold standard but are slow and expensive. A popular compromise is to let a strong LLM **judge** the answers ([Zheng et al., 2023](https://arxiv.org/abs/2306.05685)): it gets the question, the reference answer and the system's answer, and decides whether the answer is correct.

**6.1 Complete `judge`.** Write a prompt that asks the model to reply with exactly one word, `CORRECT` or `INCORRECT`, and turn the reply into `1.0` or `0.0`. Make sure that a reply like `"INCORRECT"` is not counted as correct because it *contains* the word "CORRECT".

In [ ]:
def judge(question, references, answer):
    # 1.0 if the LLM judges the answer to be correct, else 0.0
    prompt = f"""
    You are grading answers to a quiz. Decide whether the answer to the question is correct, given the reference answer(s).
    The answer may be phrased differently or be longer than the reference.

    Question: {question}
    Reference answer(s): {' | '.join(sorted(set(references)))}
    Answer: {answer}

    Reply with exactly one word: CORRECT or INCORRECT.
    """  # TODO: the question, the reference answer(s), the answer, and the instruction
    reply, _ = chat([{"role": "user", "content": prompt}], max_tokens=5)
    return float(reply.strip().upper().startswith("CORRECT"))  # TODO


print(judge("Which team won Super Bowl 50?", ["Denver Broncos"], "It was won by the Broncos from Denver."))
print(judge("Which team won Super Bowl 50?", ["Denver Broncos"], "The Carolina Panthers."))

In [ ]:
# ✅ Check judge
assert judge("Which team won Super Bowl 50?", ["Denver Broncos"], "It was won by the Broncos from Denver.") == 1.0
assert judge("Which team won Super Bowl 50?", ["Denver Broncos"], "The Carolina Panthers.") == 0.0
assert judge("What is the capital of Denmark?", ["Copenhagen"], "unanswerable") == 0.0
print("Looks good ✅")

**6.2 Compare the judge with EM and containment** on the answers of the hybrid RAG system: compute the three scores for every answer, and print the questions on which the judge disagrees with EM.

In [ ]:
answers = settings["hybrid"]
questions = [queries[i]["question"] for i in range(N_RAG)]

judge_scores = [judge(q, r, a) for q, r, a in zip(questions, references, answers)]     # TODO: a list with the judge's verdict for every answer
em_scores = [exact_match(a, r) for a, r in zip(answers, references)]        # TODO
contains_scores = [contains_answer(a, r) for a, r in zip(answers, references)]  # TODO

print(f"EM {100 * np.mean(em_scores):.1f}   contains {100 * np.mean(contains_scores):.1f}   judge {100 * np.mean(judge_scores):.1f}\n")
for q, r, a, j, e in zip(questions, references, answers, judge_scores, em_scores):
    if j != e:
        print(f"Q: {q}\n   reference: {sorted(set(r))}   answer: {a!r}   judge: {j:.0f}   EM: {e:.0f}\n")

In [ ]:
# ✅ Check the scores
assert len(judge_scores) == len(em_scores) == len(contains_scores) == N_RAG, "one score per answer"
assert set(judge_scores) <= {0.0, 1.0}, "the judge should return 0.0 or 1.0"
print("Looks good ✅")

**6.3 Can we trust the judge?** Read the disagreements: who is right, the judge or EM? Then name at least two ways in which an LLM judge can be **biased or wrong**, and one thing you would do before using an LLM judge to compare systems in your project.

---

*Your answer here:*

---

## 7. MCQ

Answer each question by writing the letter of your choice (A–D) after **Answer:**.

---

### 7.1. IDF

In BM25, which query term contributes the **least** to the score of a document that contains it?

A. A term that occurs in almost every document<br>
B. A term that occurs in a short document<br>
C. A term that occurs in only a few documents<br>
D. A term that occurs several times in the document<br>

**Answer:**

A.

---

### 7.2. Term-frequency saturation

What does the parameter $k_1$ in BM25 control?

A. How many documents are retrieved<br>
B. How much a document's length is normalised<br>
C. The weight of the query terms relative to the document terms<br>
D. How quickly additional occurrences of a term stop increasing the score<br>

**Answer:**

D.

---

### 7.3. Dense retrieval

What is the main advantage of a dense retriever over BM25?

A. It needs no training<br>
B. It is always more accurate on exact names and numbers<br>
C. It can match a query and a passage that express the same meaning in different words<br>
D. It does not need an index<br>

**Answer:**

C.

---

### 7.4. Recall@k

A RAG system puts the top 5 retrieved passages into the prompt. The retriever's Recall@5 is 0.6. What does this imply?

A. The system answers 60 % of the questions correctly<br>
B. 60 % of the retrieved passages are relevant<br>
C. The generator should be given 60 % fewer passages<br>
D. For 40 % of the questions the relevant passage is not in the prompt, so the generator cannot read the answer from it<br>

**Answer:**

D.

---

### 7.5. LLM-as-a-judge

You use GPT-style model X as a judge to compare model X with model Y, and X wins. Which concern is most specific to this setup?

A. LLM judges cannot read long answers<br>
B. LLM judges tend to prefer outputs similar to their own (self-preference bias), so the comparison may favour X<br>
C. The judge is too slow<br>
D. LLM judges always give the same score<br>

**Answer:**

B.